# The Skill Lifecycle: Discovery → Activation → Execution

Every interaction between an agent and a skill follows a structured three-phase lifecycle. Understanding this lifecycle is essential both for **skill authors** - who need to design skills that work at each phase - and for **agent architects** - who need to build systems that handle lifecycle transitions correctly.

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  DISCOVERY  │───▶│ ACTIVATION  │────▶│  EXECUTION  │
│             │     │             │     │             │
│ Find the    │     │ Load and    │     │ Agent works │
│ right skill │     │ prepare the │     │ guided by   │
│ for the task│     │ skill body  │     │ the skill   │
└─────────────┘     └─────────────┘     └─────────────┘
```

In [1]:
import re
import os
from dataclasses import dataclass, field
from typing import Annotated, Optional, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initializing the LLMs

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)
print("Model ready:", llm.model)

# Token counting: tiktoken gives a fast, consistent estimate. It is OpenAI's tokenizer
try:
    import tiktoken
    _enc = tiktoken.encoding_for_model("gpt-4o")
    def count_tokens(text: str) -> int:
        return len(_enc.encode(text))
    print("Token counter: tiktoken (approximate)")
except ImportError:
    def count_tokens(text: str) -> int:
        return len(text) // 4  # ~4 chars per token for English
    print("Token counter: character estimate")

Model ready: gpt-4o-mini
Token counter: tiktoken (approximate)


## Skill registry
We define a small registry of skills and their bodies to use throughout the notebook. In a real deployment, these would be files on disk discovered by scanning a directory tree.

In [3]:
@dataclass
class SkillRecord:
    """Full skill definition — frontmatter metadata plus the instruction body."""
    name: str
    description: str
    version: str
    allowed_tools: list[str]  # on disk this is the hyphenated `allowed-tools` YAML field
    body: str
    required_context: list[str] = field(default_factory=list)  # what must be present before activation
    tags: list[str] = field(default_factory=list)


SKILL_REGISTRY: dict[str, SkillRecord] = {
    "code-review": SkillRecord(
        name="code-review",
        description="Reviews code changes, pull requests, and diffs for correctness, security, and maintainability. Produces CRITICAL/MAJOR/MINOR severity-tiered reports.",
        version="1.0.0",
        allowed_tools=["read_file", "run_linter"],
        required_context=["code_to_review"],
        tags=["engineering", "quality"],
        body="""
You have the Code Review skill active.

## Review workflow
1. Read the entire code before forming any judgment
2. Classify every issue by severity: CRITICAL / MAJOR / MINOR
3. For each issue: line number, the problem, and a specific fix
4. Include a Strengths section and an OVERALL one-sentence verdict

## Output format
CRITICAL ISSUES: (numbered list)
MAJOR ISSUES:    (numbered list)
MINOR ISSUES:    (numbered list)
STRENGTHS:       (bullet points)
OVERALL:         (one sentence)
""",
    ),
    "git-commit-message": SkillRecord(
        name="git-commit-message",
        description="Generates conventional commit messages from staged diffs and change descriptions. Follows Conventional Commits specification.",
        version="1.0.0",
        allowed_tools=["read_file"],
        required_context=["diff_or_description"],
        tags=["engineering", "git"],
        body="""
You have the Git Commit Message skill active.

## Commit message protocol (Conventional Commits)
Format: <type>(<scope>): <description>

Types: feat, fix, docs, style, refactor, test, chore
Rules:
- Subject line ≤ 72 characters, imperative mood ('add' not 'added')
- Optional body: explain WHY, not WHAT (the diff shows what)
- Footer for breaking changes: BREAKING CHANGE: <description>

Output: just the commit message, no explanation.
""",
    ),
    "incident-response": SkillRecord(
        name="incident-response",
        description="Guides incident responders through triage, investigation, mitigation, and post-incident review for outages and production failures.",
        version="2.1.0",
        allowed_tools=["read_file", "search_logs", "send_notification"],
        required_context=["incident_description"],
        tags=["operations", "security", "oncall"],
        body="""
You have the Incident Response skill active.

## Response phases
Phase 1 — TRIAGE (first 5 min):
  Determine scope (how many users affected) and severity:
  SEV1 = all users down | SEV2 = partial outage | SEV3 = degraded performance

Phase 2 — INVESTIGATION:
  Collect logs from the 30 minutes before incident start.
  Find the first error in the timeline and correlate with recent deployments.

Phase 3 — MITIGATION:
  Apply fastest available fix: rollback, feature flag, or traffic shift.
  Confirm resolution by checking error rates and latency return to baseline.

Phase 4 — POST-INCIDENT:
  Draft incident report within 24 hours.
  Identify root cause and actionable preventive items.
""",
    ),
}

print(f"Skill registry loaded: {list(SKILL_REGISTRY.keys())}")

Skill registry loaded: ['code-review', 'git-commit-message', 'incident-response']


Each `SkillRecord` bundles the routing metadata with the instruction body, plus two fields that drive the lifecycle:
- **`required_context`** lists keys that must be present before activation - the pre-flight gate enforced in Phase 2.
- **`allowed_tools`** declares which tools the skill may drive; Phase 2 scopes the agent down to exactly this set.
- **`body`** is the instruction text, loaded into context only once the skill is activated (Tier 2 of progressive disclosure).

## Phase 1: Discovery
Discovery finds the right skill for the current task. It operates only on frontmatter metadata - no instruction bodies are loaded. Two paths exist: **explicit invocation** (user names the skill directly, e.g. `/code-review`) and **automatic routing** (the agent classifies the query against available skill descriptions).

In [4]:
def check_explicit_invocation(query: str, registry: dict[str, SkillRecord]) -> Optional[SkillRecord]:
    """Return the skill named by an explicit /skill-name prefix, if one is present."""
    # Match a leading "/token" at the very start, e.g. "/code-review ..." -> "code-review"
    match = re.match(r"^/(\S+)", query.strip())
    if match:
        return registry.get(match.group(1))  # None if the named skill does not exist
    return None


def route_with_llm(query: str, registry: dict[str, SkillRecord]) -> Optional[SkillRecord]:
    """Ask the LLM to pick the best skill for a query from the description index."""
    # Expose ONLY names + descriptions to the router — instruction bodies stay unloaded
    skill_list = "\n".join(f"  {s.name}: {s.description}" for s in registry.values())
    prompt = f"""Select the best matching skill for this user query.
Reply with just the skill name. If no skill fits, reply 'none'.

Available skills:
{skill_list}

User query: {query}
Best skill:"""
    response = llm.invoke([HumanMessage(content=prompt)])
    # Normalize the reply (strip quotes/whitespace, lowercase) before looking it up
    name = response.content.strip().strip("'\"").lower()
    return registry.get(name) if name != "none" else None

Two helpers, one routing path each:
- **`check_explicit_invocation`** matches a leading `/skill-name` with a regex and looks it up directly - no model call, so an explicit request resolves instantly.
- **`route_with_llm`** builds a name-and-description menu and asks the model to name the best fit (or `none`), exposing only metadata to the router — never the bodies.

The orchestrator below composes these into the Phase 1 contract: try the free explicit path first, fall back to LLM routing, and report which method matched.

In [5]:
def discover_skill(query: str, registry: dict[str, SkillRecord]) -> tuple[Optional[SkillRecord], str]:
    """Phase 1: route a query to a skill, preferring the cheap explicit path.

    Returns (skill, routing_method). Returns (None, "no_match") when nothing fits, which is a valid outcome — the caller then falls back to the base model.
    """
    # 1) Free path: the user named the skill directly with /skill-name
    explicit = check_explicit_invocation(query, registry)
    if explicit:
        return explicit, "explicit_invocation"

    # 2) Paid path: classify the natural-language query against the description index
    matched = route_with_llm(query, registry)
    if matched:
        return matched, "llm_routing"

    return None, "no_match"


# Exercise both routing paths, including a query that should match nothing
test_queries = [
    "Can you review this pull request?",
    "/git-commit-message staged changes: added user auth module",  # explicit prefix
    "Production is down — all users getting 500 errors",
    "What's the capital of France?",  # no skill should match
]

print("Discovery results:")
for q in test_queries:
    skill, method = discover_skill(q, SKILL_REGISTRY)
    result = skill.name if skill else "(no match)"
    print(f"  [{method:<20}] '{q[:50]:<50}' -> {result}")

Discovery results:
  [llm_routing         ] 'Can you review this pull request?                 ' -> code-review
  [explicit_invocation ] '/git-commit-message staged changes: added user aut' -> git-commit-message
  [llm_routing         ] 'Production is down — all users getting 500 errors ' -> incident-response
  [no_match            ] 'What's the capital of France?                     ' -> (no match)


Discovery correctly routes explicit invocations instantly (no LLM call) and uses LLM routing to match natural language queries. When no skill fits - like a general knowledge question - the agent falls back to base LLM behavior.

## The agent's tools
Both later phases need a concrete tool registry: activation scopes it per skill, and execution binds the scoped subset to the agent. We define four generic tools - a file reader, a linter, a log searcher, and a notifier - each doing one thing with no skill-specific logic baked in.

In [6]:
# The tools the agent can call during execution — generic, with no skill logic inside.
@tool
def read_file(path: str) -> str:
    """Read a file and return its contents."""
    files = {
        "discount.py": "def calc(price, pct):\n    return price - (price * pct)  # bug: pct=20 not 0.2\n",
        "auth.py": "def login(u, p):\n    q = f\"SELECT * FROM users WHERE u='{u}' AND p='{p}'\"\n    return db.run(q)\n",
    }
    return files.get(path, f"File not found: {path}")


@tool
def run_linter(code: str) -> str:
    """Run static analysis on Python code and return issues found."""
    issues = []
    if "price * pct" in code and "pct" in code:
        issues.append("Possible percentage vs fraction error: pct looks like an integer percentage but used directly as a multiplier")
    if "f\"SELECT" in code or "f'SELECT" in code:
        issues.append("SQL injection risk: query built with f-string interpolation")
    return "\n".join(issues) if issues else "No issues detected"


@tool
def search_logs(query: str, minutes_back: int = 30) -> str:
    """Search system logs for the given query over the past N minutes."""
    return f"[{minutes_back}min logs] Found 47 errors matching '{query}': first error at T-28min in payment-service (NullPointerException in OrderProcessor.java:142)"


@tool
def send_notification(channel: str, message: str) -> str:
    """Send a notification to a team channel (Slack, PagerDuty, etc)."""
    return f"Notification sent to #{channel}: {message}"


ALL_TOOLS = [read_file, run_linter, search_logs, send_notification]

# ALL_TOOLS is the full, unscoped registry; activation narrows it per skill
print("Tools registered:", [t.name for t in ALL_TOOLS])

Tools registered: ['read_file', 'run_linter', 'search_logs', 'send_notification']


The four tools line up with what the sample skills declare: `code-review` will be scoped to `read_file` and `run_linter`, while `incident-response` reaches for `search_logs` and `send_notification`. `ALL_TOOLS` is the full, unscoped registry that activation narrows down for each skill.

## Phase 2: Activation
Activation loads the selected skill's full body, scopes the available tools to the skill's allow-list, and runs a pre-flight check to verify required context is present before the agent starts working.

In [7]:
@dataclass
class ActivationResult:
    """Outcome of activation — success plus everything the execution phase needs."""
    success: bool
    skill: Optional[SkillRecord]
    scoped_tools: list        # tool objects permitted by this skill
    system_prompt: str        # assembled skill context (banner + body)
    failure_reason: str = ""  # populated only when success is False
    token_cost: int = 0       # tokens the assembled context will consume


def activate_skill(
    skill: SkillRecord,
    available_context: dict,
    all_tools: list,
    warn_at: int = 3000,
    hard_limit: int = 5000,
) -> ActivationResult:
    """Phase 2: load the body, scope tools, and pre-flight context — before any LLM call.

    Args:
        skill: The skill chosen during discovery.
        available_context: Keys describing what context is currently present.
        all_tools: The full tool registry to scope the skill against.
        warn_at / hard_limit: Soft and hard token budgets for the skill body.
    """
    tool_map = {t.name: t for t in all_tools}  # name -> tool object, used for scoping

    # Gate 1 — token budget. An oversized body is an authoring bug: refuse, don't truncate.
    body_tokens = count_tokens(skill.body)
    if body_tokens > hard_limit:
        return ActivationResult(
            success=False, skill=skill, scoped_tools=[], system_prompt="",
            failure_reason=f"Skill body exceeds hard limit: {body_tokens} > {hard_limit} tokens. Move content to references/.",
        )
    if body_tokens > warn_at:
        print(f"  [WARNING] Skill '{skill.name}' body is large: {body_tokens} tokens (warn at {warn_at})")

    # Gate 2 — required context. Every declared key must be present, or we stop and ask.
    missing = [r for r in skill.required_context if r not in available_context]
    if missing:
        return ActivationResult(
            success=False, skill=skill, scoped_tools=[], system_prompt="",
            failure_reason=f"Missing required context: {missing}. Please provide: {missing}",
        )

    # Gate 3 — tool scoping. Keep only declared tools that actually exist; warn on the rest.
    scoped = [tool_map[name] for name in skill.allowed_tools if name in tool_map]
    missing_tools = [name for name in skill.allowed_tools if name not in tool_map]
    if missing_tools:
        print(f"  [WARNING] Tools declared in skill but not in registry: {missing_tools}")

    # Assemble the system prompt the agent will run under: banner + permitted tools + body
    system_prompt = f"""Active skill: {skill.name} v{skill.version}
Permitted tools: {[t.name for t in scoped]}
{skill.body}"""

    return ActivationResult(
        success=True, skill=skill, scoped_tools=scoped,
        system_prompt=system_prompt, token_cost=count_tokens(system_prompt),
    )

Activation runs three gates in order, all before any expensive model call:
- **Token budget** - the body must fit the limits (warn at 3k, hard-stop at 5k); an oversized body is an authoring bug, so activation refuses rather than truncating.
- **Required context** - every key in `required_context` must be present, otherwise activation fails with a message the agent can relay to the user.
- **Tool scoping** - only declared tools that actually exist are bound; unknown names are warned about, not fatal.

On success it returns the assembled system prompt and the scoped tool list. The tests below show a clean activation and a graceful missing-context failure.

In [8]:
# Test 1 — happy path: the required code context is present
skill = SKILL_REGISTRY["code-review"]
result = activate_skill(skill, available_context={"code_to_review": True}, all_tools=ALL_TOOLS)
print(f"Activation: {'SUCCESS' if result.success else 'FAILED'}")
print(f"Scoped tools: {[t.name for t in result.scoped_tools]}")
print(f"System prompt token cost: {result.token_cost}")
print()

# Test 2 — same skill, but the required code context is missing -> graceful failure
result_fail = activate_skill(skill, available_context={}, all_tools=ALL_TOOLS)
print(f"Activation (missing context): {'SUCCESS' if result_fail.success else 'FAILED'}")
print(f"Failure reason: {result_fail.failure_reason}")

Activation: SUCCESS
Scoped tools: ['read_file', 'run_linter']
System prompt token cost: 154

Activation (missing context): FAILED
Failure reason: Missing required context: ['code_to_review']. Please provide: ['code_to_review']


## Phase 3: Execution with LangGraph
Execution is where the agent works under the active skill's guidance. We model this as a LangGraph `StateGraph` - a directed graph where each node represents a step (agent reasoning, tool execution, resource loading) and edges control which step runs next.

The graph has three nodes:
- **`agent`** - calls the LLM with skill context; decides whether to call tools or finish
- **`tools`** - executes tool calls requested by the agent
- **`END`** - reached when the agent produces a final response with no pending tool calls

In [9]:
class SkillAgentState(TypedDict):
    """Minimal execution state — only the two fields the graph nodes actually read."""
    messages: Annotated[list, add_messages]  # history; add_messages appends new turns
    skill_system_prompt: str                 # skill context, re-injected each agent turn


def build_skill_agent_graph(activation: ActivationResult):
    """Phase 3: build a compiled agent->tools->agent graph scoped to the active skill.

    Returns a compiled LangGraph ready to invoke with an initial state.
    """
    # Bind ONLY the scoped tools — the agent has no way to call anything outside the skill
    scoped_llm = llm.bind_tools(activation.scoped_tools)
    tool_node = ToolNode(activation.scoped_tools)

    def agent_node(state: SkillAgentState) -> dict:
        """Reasoning step: prepend the skill context, then let the LLM choose its next move."""
        # The skill prompt leads every turn so guidance persists across the loop
        messages = [SystemMessage(content=state["skill_system_prompt"])] + state["messages"]
        response = scoped_llm.invoke(messages)
        return {"messages": [response]}  # add_messages merges this into the running history

    def should_continue(state: SkillAgentState) -> str:
        """Loop to the tools node while the agent is still requesting tool calls."""
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        return END  # agent answered with plain text -> finish

    # Wire the ReAct cycle: agent -> (tools -> agent)* -> END
    graph = StateGraph(SkillAgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)
    graph.set_entry_point("agent")
    graph.add_conditional_edges("agent", should_continue)  # agent decides: tools or END
    graph.add_edge("tools", "agent")                       # after tools run, hand back to agent
    return graph.compile()


print("LangGraph execution graph builder ready")

LangGraph execution graph builder ready


The graph is the standard ReAct loop, scoped by the skill:
- **State** carries just the message history (merged by the `add_messages` reducer) and the skill system prompt re-injected on every agent turn - the decorative fields of a fuller agent are intentionally omitted.
- **`agent_node`** prepends that skill context, then calls an LLM bound only to the scoped tools.
- **`should_continue`** routes to the `tools` node whenever the last message carries tool calls, and to `END` once the agent answers in plain text.
- **Edges** form the cycle `agent → tools → agent`, repeating until no tool call remains.

## End-to-end demo: Full lifecycle
Now we run all three phases in sequence for a realistic task: reviewing code for bugs and security issues. We'll trace each phase explicitly so the lifecycle is visible.

In [10]:
def run_skill_lifecycle(user_query: str, registry: dict[str, SkillRecord], available_context: dict) -> str:
    """Run Discovery -> Activation -> Execution in order, tracing each phase.

    Each phase runs only if the previous one succeeded; any failure returns early with a clear, user-facing message instead of proceeding blindly.
    """
    print("\n" + "=" * 60)
    print(f"Query: {user_query[:80]}")
    print("=" * 60)

    # Phase 1 — DISCOVERY: route the query, or fall back to the base model on no match
    print("\n[Phase 1 — DISCOVERY]")
    skill, routing_method = discover_skill(user_query, registry)
    if skill is None:
        print("  No skill matched. Falling back to base LLM.")
        return llm.invoke([HumanMessage(content=user_query)]).content
    print(f"  Matched skill: '{skill.name}' via {routing_method}")
    print(f"  Declared required context: {skill.required_context}")

    # Phase 2 — ACTIVATION: load body, scope tools, verify required context is present
    print("\n[Phase 2 — ACTIVATION]")
    activation = activate_skill(skill, available_context, ALL_TOOLS)
    if not activation.success:
        print(f"  Activation failed: {activation.failure_reason}")
        return f"I need more information before I can start: {activation.failure_reason}"
    print(f"  Skill loaded: {activation.token_cost} tokens in context")
    print(f"  Scoped tools: {[t.name for t in activation.scoped_tools]}")

    # Phase 3 — EXECUTION: run the skill-guided LangGraph agent loop to completion
    print("\n[Phase 3 — EXECUTION]")
    agent_graph = build_skill_agent_graph(activation)
    initial_state: SkillAgentState = {
        "messages": [HumanMessage(content=user_query)],
        "skill_system_prompt": activation.system_prompt,
    }
    final_state = agent_graph.invoke(initial_state)

    # The answer is the last assistant message that has no pending tool calls
    for msg in reversed(final_state["messages"]):
        if isinstance(msg, AIMessage) and not getattr(msg, "tool_calls", []):
            return msg.content
    return "(No response generated)"

`run_skill_lifecycle` is just the three phases wired in sequence, with an early return whenever a phase cannot proceed. We run it now on a real review task that supplies the required `code_to_review` context, so all three phases execute end to end.

In [11]:
SAMPLE_CODE = """
def calculate_discount(price, discount_pct):
    return price - (price * discount_pct)  # bug: pct=20 treated as fraction

def login(username, password):
    query = f\"SELECT * FROM users WHERE u='{username}' AND p='{password}'\"
    return db.execute(query)  # SQL injection
"""

response = run_skill_lifecycle(
    user_query=f"Review this code for bugs and security issues:\n{SAMPLE_CODE}",
    registry=SKILL_REGISTRY,
    available_context={"code_to_review": True},
)

print("\n" + "─" * 60)
print("FINAL RESPONSE:")
print("─" * 60)
print(response)


Query: Review this code for bugs and security issues:

def calculate_discount(price, di

[Phase 1 — DISCOVERY]
  Matched skill: 'code-review' via llm_routing
  Declared required context: ['code_to_review']

[Phase 2 — ACTIVATION]
  Skill loaded: 154 tokens in context
  Scoped tools: ['read_file', 'run_linter']

[Phase 3 — EXECUTION]

────────────────────────────────────────────────────────────
FINAL RESPONSE:
────────────────────────────────────────────────────────────
CRITICAL ISSUES:
1. **Line 4**: SQL injection risk - The query is built using f-string interpolation, which allows for SQL injection if user input is not properly sanitized. 
   - **Fix**: Use parameterized queries to prevent SQL injection, e.g., `query = "SELECT * FROM users WHERE u=%s AND p=%s"` and pass the parameters separately.

MAJOR ISSUES:
1. **Line 2**: Incorrect handling of discount percentage - The function assumes `discount_pct` is a decimal fraction, but if a user inputs a percentage like 20, it will not wo

The lifecycle trace makes each phase explicit:
- **Discovery** found the right skill from the unstructured query
- **Activation** loaded the instruction body, scoped tools, and verified required context was present
- **Execution** ran the LangGraph loop: agent reasoned with skill guidance, called `run_linter`, and produced a structured review

Each phase only executes if the previous one succeeded - failures surface immediately with clear messages.

## Skill chaining
Complex multi-phase tasks often require more than one skill. **Skill chaining** runs skills sequentially, passing the output of each phase forward as input to the next. A research-analysis-report pipeline is a classic example: the research skill gathers information, the data analysis skill processes it, and the report writing skill formats the output.

In [12]:
@dataclass
class ChainStep:
    """One step in a skill chain: which skill runs and which prior outputs it consumes."""
    phase: str               # logical phase name, e.g. 'gather', 'review', 'report'
    skill_name: str          # the skill to activate for this phase
    input_phases: list[str]  # earlier phases whose output is fed in as context


def run_skill_chain(task: str, chain: list[ChainStep], registry: dict[str, SkillRecord]) -> dict[str, str]:
    """Run skills in sequence, feeding selected prior outputs into each next phase.

    Returns a mapping of phase name -> that phase's output text.
    """
    results: dict[str, str] = {}  # accumulates each phase's output for downstream steps

    for step in chain:
        print(f"\n{'-'*50}\nChain phase: {step.phase} (skill: {step.skill_name})")
        skill = registry.get(step.skill_name)
        if not skill:
            print(f"  [SKIP] Skill '{step.skill_name}' not found")
            continue

        # Stitch in only the outputs of the prior phases this step declared as inputs
        prior = ""
        for ip in step.input_phases:
            if ip in results:
                prior += f"\n\n[Output from {ip} phase]:\n{results[ip]}"
        phase_prompt = f"{task}{prior}\n\nYour task for this phase: {step.phase}"

        # Reuse Phase 2 to activate this step's skill (treat its required context as supplied)
        activation = activate_skill(skill, {r: True for r in skill.required_context}, ALL_TOOLS)
        if not activation.success:
            print(f"  [FAILED] {activation.failure_reason}")
            continue

        # Run the skill-guided model and store the output for later phases to consume
        response = llm.invoke([
            SystemMessage(content=activation.system_prompt),
            HumanMessage(content=phase_prompt),
        ])
        results[step.phase] = response.content
        print(f"  Phase output preview: {response.content[:120]}...")

    return results

The chain runner keeps composition explicit and stateless between skills:
- **`ChainStep`** declares, per phase, which skill runs and which earlier phases' outputs feed it - dependencies are data, not hard-coded control flow.
- **`run_skill_chain`** activates each step's skill in turn (reusing Phase 2), stitches in only the requested prior outputs, runs the skill-guided model, and records the result for downstream phases.
- **Isolation** - each skill sees only the task plus its declared inputs, so phases stay independently testable and swappable.

To make the pipeline concrete we add two more skills - a research assistant and a report writer - alongside the existing `code-review` skill, then define a three-step chain that gathers context, reviews against it, and writes a synthesized report.

In [13]:
SKILL_REGISTRY["research-assistant"] = SkillRecord(
    name="research-assistant",
    description="Researches a topic by gathering relevant facts, data, and context from available sources.",
    version="1.0.0",
    allowed_tools=["read_file"],
    required_context=["research_topic"],
    body="""
You have the Research Assistant skill active.
Your goal: gather comprehensive facts and data on the given topic.
Output: bullet-point summary of key findings with sources cited.
""",
)

SKILL_REGISTRY["report-writer"] = SkillRecord(
    name="report-writer",
    description="Synthesizes research findings and analysis into a structured, professional report.",
    version="1.0.0",
    allowed_tools=[],
    required_context=["report_content"],
    body="""
You have the Report Writer skill active.
Your goal: synthesize the provided findings into a clear, structured report.
Format: Executive Summary → Key Findings → Recommendations → Next Steps
Tone: professional, concise, actionable.
""",
)

# Define the chain: Research → Code Review → Report
ANALYSIS_CHAIN = [
    ChainStep(
        phase="gather",
        skill_name="research-assistant",
        input_phases=[],
    ),
    ChainStep(
        phase="review",
        skill_name="code-review",
        input_phases=["gather"],  # code review gets the research context
    ),
    ChainStep(
        phase="report",
        skill_name="report-writer",
        input_phases=["gather", "review"],  # report synthesizes both phases
    ),
]
print("Chain defined:", [s.phase for s in ANALYSIS_CHAIN])

Chain defined: ['gather', 'review', 'report']


Running the chain streams each phase's activation and output preview in order; the final phase's output is the synthesized report, which we print on its own afterward.

In [14]:
print("Running skill chain: Research -> Code Review -> Report")
chain_results = run_skill_chain(
    task="Assess the security posture of our authentication module",
    chain=ANALYSIS_CHAIN,
    registry=SKILL_REGISTRY,
)
print(f"\nCompleted phases: {list(chain_results.keys())}")

Running skill chain: Research -> Code Review -> Report

--------------------------------------------------
Chain phase: gather (skill: research-assistant)
  Phase output preview: To assess the security posture of your authentication module, I will need to gather relevant information and best practi...

--------------------------------------------------
Chain phase: review (skill: code-review)
  Phase output preview: CRITICAL ISSUES:
1. **Lack of Multi-Factor Authentication (MFA)**: The absence of MFA exposes the system to unauthorized...

--------------------------------------------------
Chain phase: report (skill: report-writer)
  Phase output preview: # Executive Summary

The assessment of the authentication module's security posture reveals critical vulnerabilities tha...

Completed phases: ['gather', 'review', 'report']


In [15]:
print("\n" + "=" * 60)
print("FINAL REPORT (from 'report' phase):")
print("=" * 60)
print(chain_results.get("report", "(no report generated)"))


FINAL REPORT (from 'report' phase):
# Executive Summary

The assessment of the authentication module's security posture reveals critical vulnerabilities that could expose the system to unauthorized access and data breaches. Key areas of concern include the lack of Multi-Factor Authentication (MFA), weak password management practices, insecure session management, and the absence of secure transmission protocols. Addressing these issues is essential to enhance the overall security of the authentication module and protect user data.

# Key Findings

### Critical Issues
1. **Lack of Multi-Factor Authentication (MFA)**: The absence of MFA increases the risk of unauthorized access through compromised passwords.
2. **Weak Password Management**: Failure to enforce strong password policies heightens the risk of password-related attacks.

### Major Issues
1. **Insecure Session Management**: Unsecured session tokens are vulnerable to hijacking.
2. **Lack of Secure Transmission**: Not utilizing H

Skill chaining composes simple, focused skills into sophisticated multi-phase workflows. Each skill only knows about its own task - the chain orchestrates the handoff. This separation means we can swap out the report-writer skill for a different format (e.g. a Slack-post writer) without touching the research or review skills.

## Failure modes and graceful handling
Production agents encounter failures. The lifecycle design anticipates the most common ones and handles each with a specific recovery strategy.

In [16]:
print("Failure mode demonstrations (1-2):\n")

# Failure 1 — no skill matched: discovery returns None and we use the base model
print("1. No skill matched")
skill, method = discover_skill("What is the meaning of life?", SKILL_REGISTRY)
if skill is None:
    fallback = llm.invoke([HumanMessage(content="What is the meaning of life?")])
    print(f"   -> Graceful fallback to base LLM: {fallback.content[:100]}...")
print()

# Failure 2 — required context missing: activation pre-flight stops before any work
print("2. Required context missing")
result = activate_skill(SKILL_REGISTRY["code-review"], available_context={}, all_tools=ALL_TOOLS)
print(f"   -> Failure reason: {result.failure_reason}")
print("   -> Recovery: ask the user to share the code or file path")

Failure mode demonstrations (1-2):

1. No skill matched
   -> Graceful fallback to base LLM: The meaning of life is a profound and philosophical question that has been explored by thinkers, the...

2. Required context missing
   -> Failure reason: Missing required context: ['code_to_review']. Please provide: ['code_to_review']
   -> Recovery: ask the user to share the code or file path


The first two failures surface at discovery and at the activation pre-flight. The remaining two occur deeper in activation - when the body is too large, or when a declared tool is missing:

In [17]:
print("Failure mode demonstrations (3-4):\n")

# Failure 3 — body over the hard token limit: activation refuses to load it
print("3. Skill body too large (over hard limit)")
oversized_skill = SkillRecord(
    name="oversized", description="A skill with too much body content",
    version="1.0.0", allowed_tools=[], required_context=[],
    body="This is a very long skill body. " * 600,  # ~6000 tokens
)
result_large = activate_skill(oversized_skill, available_context={}, all_tools=ALL_TOOLS)
print(f"   -> Failure reason: {result_large.failure_reason}")
print("   -> Recovery: move body content to references/ and load it on demand")
print()

# Failure 4 — a declared tool does not exist: activation warns but still proceeds
print("4. Tool declared in skill but not registered")
skill_missing_tool = SkillRecord(
    name="test", description="Test skill", version="1.0.0",
    allowed_tools=["read_file", "nonexistent_tool"], required_context=[], body="Test body",
)
result_missing = activate_skill(skill_missing_tool, available_context={}, all_tools=ALL_TOOLS)
print(f"   -> Activation succeeds, scoped to existing tools only: {[t.name for t in result_missing.scoped_tools]}")

Failure mode demonstrations (3-4):

3. Skill body too large (over hard limit)
  [WARNING] Skill 'oversized' body is large: 4801 tokens (warn at 3000)
   -> Failure reason: 
   -> Recovery: move body content to references/ and load it on demand

4. Tool declared in skill but not registered
  [WARNING] Tools declared in skill but not in registry: ['nonexistent_tool']
   -> Activation succeeds, scoped to existing tools only: ['read_file']


Each failure mode has a distinct, recoverable response:

| Failure | Detection point | Recovery |
|---------|----------------|----------|
| No skill matched | Discovery | Fallback to base LLM - still useful |
| Required context missing | Activation pre-flight | Ask user for missing input |
| Skill body too large | Activation token check | Hard stop - fix skill, don't proceed |
| Tool not in registry | Activation tool scoping | Warn and proceed with available tools |

No failure silently degrades into unpredictable behavior - each is surfaced with a clear message and a defined recovery path.